# Notebook 02 — Link Kaggle Healthcare CSV to MPI

**Goal:** Assign each of the 10,000 Kaggle admission rows a `patient_id` from the MPI.

**Linking logic:**
- Match by gender (exact)
- Match by age (±5 years)
- If no match found → fallback to same gender random assignment

**Output:** `data_preparation/linked/admissions_linked.csv`

## 1. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import random
import os

KAGGLE_PATH = "../raw/kaggle_healthcare/healthcare_dataset.csv"
MPI_PATH    = "../linked/patients_master.csv"
OUTPUT_DIR  = "../linked/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths OK")

Paths OK


## 2. Load Data

In [2]:
kaggle = pd.read_csv(KAGGLE_PATH)
mpi    = pd.read_csv(MPI_PATH)

print(f"Kaggle records : {len(kaggle)} rows")
print(f"MPI patients   : {len(mpi)} rows")
print()
print("Kaggle columns:")
print(kaggle.columns.tolist())
print()
print("Kaggle sample:")
kaggle.head(3)

Kaggle records : 55500 rows
MPI patients   : 1163 rows

Kaggle columns:
['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']

Kaggle sample:


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal


## 3. Normalize Gender Values

In [3]:
# Kaggle uses 'Male'/'Female', MPI uses 'M'/'F'
# Normalize Kaggle gender to match MPI format
kaggle["gender_normalized"] = kaggle["Gender"].map({"Male": "M", "Female": "F"})

print("Kaggle gender distribution:")
print(kaggle["gender_normalized"].value_counts())
print()
print("MPI gender distribution:")
print(mpi["GENDER"].value_counts())

Kaggle gender distribution:
gender_normalized
M    27774
F    27726
Name: count, dtype: int64

MPI gender distribution:
GENDER
F    616
M    547
Name: count, dtype: int64


## 4. Build Gender-Age Lookup from MPI

In [4]:
# Group MPI patients by gender for fast lookup
mpi_male   = mpi[mpi["GENDER"] == "M"][["patient_id", "age"]].reset_index(drop=True)
mpi_female = mpi[mpi["GENDER"] == "F"][["patient_id", "age"]].reset_index(drop=True)

print(f"MPI male patients   : {len(mpi_male)}")
print(f"MPI female patients : {len(mpi_female)}")

MPI male patients   : 547
MPI female patients : 616


## 5. Link Each Kaggle Row to a Patient ID

In [5]:
def find_patient_id(gender, age, age_tolerance=5):
    """
    Find a matching patient_id from the MPI.
    Strategy:
      1. Same gender + age within tolerance → pick randomly from matches
      2. Same gender only (fallback) → pick randomly from same gender pool
    Returns (patient_id, match_type)
    """
    pool = mpi_male if gender == "M" else mpi_female

    # Try age match within tolerance
    age_matched = pool[(pool["age"] >= age - age_tolerance) &
                       (pool["age"] <= age + age_tolerance)]

    if len(age_matched) > 0:
        return random.choice(age_matched["patient_id"].tolist()), "gender_age_match"
    else:
        # Fallback: same gender, any age
        return random.choice(pool["patient_id"].tolist()), "gender_only_match"


# Apply to every row
print("Linking Kaggle rows to patient IDs...")
results = kaggle.apply(
    lambda row: find_patient_id(row["gender_normalized"], row["Age"]),
    axis=1
)

kaggle["patient_id"]  = results.apply(lambda x: x[0])
kaggle["match_type"]  = results.apply(lambda x: x[1])

print("Done.")
print()
print("Match type distribution:")
print(kaggle["match_type"].value_counts())

Linking Kaggle rows to patient IDs...
Done.

Match type distribution:
match_type
gender_age_match    55500
Name: count, dtype: int64


## 6. Clean Up & Final Structure

In [6]:
# Drop the temporary normalized gender column
kaggle.drop(columns=["gender_normalized"], inplace=True)

# Reorder columns — patient_id and match_type first
cols = ["patient_id", "match_type"] + [c for c in kaggle.columns if c not in ["patient_id", "match_type"]]
kaggle = kaggle[cols]

print("Final columns:")
print(kaggle.columns.tolist())
print()
kaggle.head(5)

Final columns:
['patient_id', 'match_type', 'Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']



,patient_id,match_type,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,1d751b54-8ff3-4fe6-91df-751a230c94d7,gender_age_match,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,90a958eb-68a4-aa61-85e0-e9f20b083d94,gender_age_match,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,07fc8824-40ff-4c97-898d-f906bc6f2fd3,gender_age_match,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,d637aa7b-6fbc-ef23-c55a-d9c33b3d376d,gender_age_match,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,e3ca21de-75e7-39fa-b0fa-fc4341e1c54e,gender_age_match,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


## 7. Quality Check

In [7]:
print("=== Admissions Linking Quality Check ===")
print(f"Total admission records     : {len(kaggle)}")
print(f"Unique patients assigned    : {kaggle['patient_id'].nunique()}")
print(f"Patients with 0 admissions  : {len(mpi) - kaggle['patient_id'].nunique()}")
print()
print("Match type distribution:")
print(kaggle["match_type"].value_counts())
print()
print("Admissions per patient (stats):")
admissions_per_patient = kaggle.groupby("patient_id").size()
print(admissions_per_patient.describe())
print()
print("Top 5 most admitted patients:")
print(admissions_per_patient.sort_values(ascending=False).head())
print()
print("Medical condition distribution:")
print(kaggle["Medical Condition"].value_counts())

=== Admissions Linking Quality Check ===
Total admission records     : 55500
Unique patients assigned    : 998
Patients with 0 admissions  : 165

Match type distribution:
match_type
gender_age_match    55500
Name: count, dtype: int64

Admissions per patient (stats):
count    998.000000
mean      55.611222
std       21.076999
min        1.000000
25%       46.000000
50%       57.000000
75%       67.000000
max      131.000000
dtype: float64

Top 5 most admitted patients:
patient_id
22e14c09-32f4-0b73-36e3-b38c967afd1b    131
7f55478e-e6c6-6bef-10b3-c9be22b3253b    125
9fe87de4-271d-0775-9319-858ac7f42a6f    124
25e2030b-e0c7-3b91-d434-57693cb72659    120
f56514f4-c048-cf96-796a-d0ec65dbd6e4    112
dtype: int64

Medical condition distribution:
Medical Condition
Arthritis       9308
Diabetes        9304
Hypertension    9245
Obesity         9231
Cancer          9227
Asthma          9185
Name: count, dtype: int64


## 8. Save Output

In [8]:
output_path = os.path.join(OUTPUT_DIR, "admissions_linked.csv")
kaggle.to_csv(output_path, index=False)

print(f"Saved → {output_path}")
print(f"Shape  : {kaggle.shape}")

Saved → ../linked/admissions_linked.csv
Shape  : (55500, 17)


In [9]:
# Check the raw file before any processing
raw = pd.read_csv("../raw/kaggle_healthcare/healthcare_dataset.csv")
print(f"Raw file rows : {len(raw)}")
print(f"Raw file cols : {raw.shape[1]}")
print()
print(raw.head(3))

Raw file rows : 55500
Raw file cols : 15

            Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31   
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22   

             Doctor         Hospital Insurance Provider  Billing Amount  \
0     Matthew Smith  Sons and Miller         Blue Cross    18856.281306   
1   Samantha Davies          Kim Inc           Medicare    33643.327287   
2  Tiffany Mitchell         Cook PLC              Aetna    27955.096079   

   Room Number Admission Type Discharge Date   Medication  Test Results  
0          328         Urgent     2024-02-02  Paracetamol        Normal  
1          265      Emergency     2019-08-26    Ibuprofen  Inconclusive  
2          205      Emergency     2022-10-07      Aspirin        Normal  
